In [ ]:
import flow
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import shutil
import matplotlib.pyplot as plt
from flow import FlowProject, directives
from pymser import pymser
import warnings
import panedr

In [ ]:
from utils.molec_class_files import esolvs
dict = esolvs.make_dict()
for i, j in dict.items():
    nmols = 20000// j.n_atoms
    # print(j.molecular_weight, j.expt_liq_density.values())
    V = (nmols*j.molecular_weight*1e27)/(max(list(j.expt_liq_density.values())) * 1000* 6.022*1e23)
    # print(j.bounds_sig)
    boxy =13.2*np.max(j.bounds_sig)
    boxz = V/boxy**2

    if boxz < boxy:
        #Calculate the number of molecules needed such that V^(1/3) = 13.2*np.max(j.bounds_sig)
        V_ratio = (13.2*np.max(j.bounds_sig))**3/V
        nmols1 = int(nmols*V_ratio)
        boxy = boxz = V**(1/3)
    else:
        nmols1 = nmols

    box3 = V**(1/3)
    print(i, boxy, boxz, box3, nmols, nmols1)

In [ ]:
file = "example_output_files/sft_calc.edr"
edr = panedr.edr_to_df(file)

In [ ]:
edr.tail()

In [ ]:
dt = edr["Time"].values
print(dt)
nsteps = 70000000
tot_rows = 70000
prod_step = 10000
nprod = tot_rows*prod_step
print(nprod)
print(nsteps/tot_rows)
print(nsteps/prod_step)

In [ ]:
len(edr)

In [ ]:
np.mean(edr["Pressure"].values)

In [ ]:
# Get the density and volume data
sim_name = "sft_calc"
# file = "example_output_files/" + sim_name + ".edr"
# file = "density_iters/runs/workspace/14d22455341f5c2c9b0ebbc9ba0a905b/inter_prod.edr"
# file = "density_iters/runs/workspace/8f253dd2fee918dfef9de6227ed0d94d/npt_prod.edr"
file = "density_iters/runs_npzzat_rect/workspace/538a482f70e1c63c2470f0d3c8c1919e/inter_eq.edr"
# file = "example_output_files/sft_calc.edr"
df_all = panedr.edr_to_df(file)
df = df_all
# file1 = "example_output_files/prd_npt.edr"
# df_all1 = panedr.edr_to_df(file)
# df_all2 = panedr.edr_to_df(file1)
# df_all2["Time"] = df_all2["Time"] + df_all1["Time"].iloc[-1]
# df_all = pd.concat([df_all1, df_all2], ignore_index=True)
print(df_all.columns)
eq_data_dict = {}
property = "Total Energy"
if property in df_all.columns:
    df = df_all[["Time", property]].copy()
    print(df)


# elif property in ["Total Energy", "Volume", "Pressure", "#Surf*SurfTen"]:
    # command = "source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC"
    # subprocess.run(command, shell=True, check=True)
    # print(os.path.exists(os.path.join("example_output_files", f"{sim_name}.edr")))
    # command = (
    #     f"source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC && module load gromacs && gmx_d energy -f example_output_files/{sim_name}.edr -s example_output_files/{sim_name}.tpr -o {sim_name}_{property}.xvg"
    # )
    # subprocess.run(command, input=f"{property}\n", text=True, check=True, shell = True)
    # prop_data = np.loadtxt(
    #     sim_name + "_" + property + ".xvg", comments=["#", "@"]
    # )

    # df = pd.DataFrame(prop_data)

# time_data = df.iloc[:int(len(df)*0.7), 0].values
# property_data = df.iloc[:int(len(df)*0.7), 1].values
time_data = df.iloc[:, 0].values
property_data = df.iloc[:, 1].values
print(np.mean(property_data))
plt.plot(time_data, property_data)
print(len(time_data))
eq_col_file = sim_name + "_" + property + ".csv"
eq_data_dict[property] = {
    "data": property_data,
    "time_data": time_data,
    "file": eq_col_file,
}

In [ ]:
def calc_mass_dens(density):
    #Find the region attributed to liquid density
    mass_dens_z = density[:, 1]
    find_liq_slab = find_bulk_liq_index(density)
    range_for_liq_dens = find_liq_slab[0]
    range_org_liq = find_liq_slab[1]
    median_dens_liq = find_liq_slab[3]
    #Calculate the density
    prop_vals = mass_dens_z[range_for_liq_dens] #kg/m^3
    plt.scatter(density[:, 0], density[:, 1], label='ES', color='blue')
    plt.axhline(y=median_dens_liq, color='g', linestyle='--', label= 'Median')
    plt.xlabel('Z (nm)')
    plt.ylabel('Density (kg/m^3)')
    med_dens_pt = find_liq_slab[4]
    
    plt.scatter(density[range_for_liq_dens, 0], density[range_for_liq_dens, 1], label='ES', color='red')
    # plt.scatter(density[range_org_liq, 0], density[range_org_liq, 1], label='ES', color='green')
    plt.scatter(density[med_dens_pt, 0], density[med_dens_pt, 1], label='ES', color='orange')
    plt.show()
    print(np.mean(prop_vals))
    print(max(density[range_for_liq_dens, 0]), min(density[range_for_liq_dens, 0]))
    return prop_vals
from scipy.signal import find_peaks
from findpeaks import findpeaks


def find_bulk_liq_index(density):
    #Use np.diff to approximate the 1st derivative
    ES_numdens_z = density[:,1]
    x = density[:,0]
    # dy = np.diff(ES_numdens_z)
    dy = np.gradient(ES_numdens_z, x)
    # dy = np.gradient(np.gradient(ES_numdens_z, x), x)
    # print(dy)
    # plt.scatter(np.arange(0,len(dy),1), dy, label='ES', color='blue')
    # plt.scatter(results['x'].iloc[interfaces], results['y'].iloc[interfaces], label='ES', color='red')
    #Use findpeaks to find the peaks and valleys, interpolating to get as close as possible
    fp = findpeaks(lookahead=1, interpolate=10, verbose=0)
    results = fp.fit(dy)["df_interp"]

    all_peaks = results[results['peak'] | results['valley']].index.values
    #get the highest peak and the lowest valley, these are the interfaces
    y_vals = results['y'].iloc[all_peaks]
    peak_index = all_peaks[np.argmax(y_vals)]
    valley_index = all_peaks[np.argmin(y_vals)]

    interfaces = [peak_index, valley_index]

    #Divide the indices by 10 to get the correct index for the density based on interpolation
    interfaces = [int(i/10) for i in interfaces]

    #Get the range of indices for the bulk liquid density regardless of whether interfaces have shifted
    if interfaces[0] < interfaces[1]:
        range_org_liq = list(range(interfaces[0], interfaces[1] + 1))
    else:
        range_org_liq = list(range(interfaces[0], len(results)//10)) + list(range(0, interfaces[1] + 1))

    if_r = [peak_index, valley_index]

    # if if_r[0] < if_r[1]:
    #     range_org_liq1 = list(range(if_r[0], if_r[1] + 1))
    # else:
    #     range_org_liq1 = list(range(if_r[0], len(results))) + list(range(0, if_r[1] + 1))

    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[all_peaks]*max(x)/(len(x)*10), results['y'].iloc[all_peaks], label='ES', color='red')
    # plt.scatter(results['x'].iloc[if_r[0]]*max(x)/(len(x)*10), results['y'].iloc[if_r[0]], label='ES', color='green')
    # plt.scatter(results['x'].iloc[if_r[1]]*max(x)/(len(x)*10), results['y'].iloc[if_r[1]], label='ES', color='green')
    
    # plt.xlabel('Z (nm)')
    # plt.ylabel('d_rho/dz (kg/m^3/nm)')
    # plt.show()

    # #Divide the indices by 10 to get the correct index for the density based on interpolation
    # interfaces = [int(i/10) for i in if_r]
    # range_org_liq = [int(i/10) for i in range_org_liq1]
    median_dens_liq = np.median(ES_numdens_z[range_org_liq])
    differences = np.abs(ES_numdens_z - median_dens_liq)
    med_dens_pt = np.argmin(differences)
    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[range_org_liq*10]*max(x)/(len(x)*10), results['y'].iloc[range_org_liq*10], label='ES', color='red')

    
    # plt.xlabel('Z (nm)')
    # plt.ylabel('d_rho/dz (kg/m^3/nm)')
    # plt.show()

    # Find the first and last index where density >= median
    valid_idx = np.where(ES_numdens_z[range_org_liq] >= median_dens_liq)[0]
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1  # +1 to include the last valid index
        range_for_liq_dens = range_org_liq[start:end]

        full_range = range_org_liq[start:end]
        print(len(range_org_liq), len(full_range), len(range_org_liq)/len(full_range))
        cutoff = int(len(full_range) * 0.85)
        trim = (len(full_range) - cutoff) // 2
        range_for_liq_dens = full_range[trim:trim + cutoff]

    else:
        range_for_liq_dens = np.array([], dtype=int)

    median_dens_liq = np.median(ES_numdens_z[range_for_liq_dens])
    differences = np.abs(ES_numdens_z - median_dens_liq)
    # med_dens_pt = np.argmin(differences)
    # print(range_org_liq)
    # range_for_liq_dens = range_org_liq
    # print(x[max(range_for_liq_dens)], x[min(range_for_liq_dens)])

    return range_for_liq_dens, range_org_liq, ES_numdens_z, median_dens_liq, med_dens_pt


In [ ]:
# density = np.loadtxt("density_iters/runs/workspace/14d22455341f5c2c9b0ebbc9ba0a905b/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/c4e7cea1aaf17ab3c761d6d98f531e6e/inter_prod_density.xvg", comments=["#", "@"])
# file = "density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg"
# density_org = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
means = []
#5dcc2f1567e58231be94b7091c8ebdd7,1feec85b78e61abf7d489be38b888430, 0ba3df2e70d9dc80d91472c528393263, 1f489249dd7f23f12833e7230f16fa4f, a4ccdf65689e3f5c229bd9960afa9cab
for r in [1,2,3]:
    # density = np.loadtxt(f"density_iters/runs_npzzat/workspace/0ba3df2e70d9dc80d91472c528393263/inter_prod_density_{r}.xvg", comments=["#", "@"])
    density = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_{r}.xvg", comments=["#", "@"])
    prop_vals = calc_mass_dens(density)
    # density = np.loadtxt(file, comments=["#", "@"])
    mean = np.mean(prop_vals)
    means.append(mean)
print(means)
print(np.var(means)/2)

In [ ]:
#1feec85b78e61abf7d489be38b888430, 0ba3df2e70d9dc80d91472c528393263
density = np.loadtxt(f"density_iters/runs_npzzat/workspace/0ba3df2e70d9dc80d91472c528393263/inter_prod_density.xvg", comments=["#", "@"])
density = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_all.xvg", comments=["#", "@"])
prop_vals = calc_mass_dens(density)
# density = np.loadtxt(file, comments=["#", "@"])
mean = np.mean(prop_vals)
print(mean)

In [ ]:
# density = np.loadtxt("density_iters/runs/workspace/14d22455341f5c2c9b0ebbc9ba0a905b/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/c4e7cea1aaf17ab3c761d6d98f531e6e/inter_prod_density.xvg", comments=["#", "@"])
# file = "density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg"
density_org = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
# density = np.loadtxt("R125_298_ex/sft_calc_density.xvg", comments=["#", "@"])
density_org = np.loadtxt(f"density_iters/runs_npzzat/workspace/0ba3df2e70d9dc80d91472c528393263/inter_prod_density.xvg", comments=["#", "@"])


n = density_org.shape[0]

for r in np.arange(0.05, 0.95, 0.05):#[0.25, 0.33, 0.5, 0.66, 0.75]:
    k = int(n * r)  # Size of the first third

    density = density_org.copy()
    # Rearranged array: from k to end, then first k rows
    density[:, 1] = np.concatenate((density_org[k:, 1], density_org[:k, 1]))


    plt.scatter(density[:, 0], density[:, 1])
    plt.xlabel('Z (nm)')
    plt.ylabel('Density (kg/m^3)')
    plt.title('Density Profile')
    plt.show()

    prop_vals = calc_mass_dens(density)

print(prop_vals)


In [ ]:
def plot_res_pymser(t_col, eq_col, results, name):
    fig, [ax1, ax2] = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [2, 1]}, sharey=True
    )

    ax1.set_ylabel(name, color="black", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Time (ns)", fontsize=14, fontweight="bold")

    ax1.plot(t_col, eq_col, label="Raw data", color="blue")

    ax1.plot(
        t_col[results["t0"] :],
        results["equilibrated"],
        label="Equilibrated data",
        color="red",
    )

    ax1.plot(
        [0, t_col[-1]],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax1.fill_between(
        t_col,
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    # ax1.set_yticks(np.arange(eq_col.min(), eq_col.max(), eq_col.max()/10))
    ax1.set_xlim(t_col.min(), t_col.max())
    ax1.tick_params(axis="y", labelcolor="black")

    ax1.grid(alpha=0.3)
    ax1.legend()

    ax2.hist(
        eq_col,
        orientation="horizontal",
        bins=30,
        edgecolor="blue",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    bin_red = 10
    ax2.hist(
        results["equilibrated"],
        orientation="horizontal",
        bins=bin_red,
        edgecolor="red",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    ymax = int(ax2.get_xlim()[-1])

    ax2.plot(
        [0, ymax],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax2.fill_between(
        range(ymax),
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    ax2.set_xlim(0, ymax)

    ax2.grid(alpha=0.5, zorder=1)

    fig.set_size_inches(9, 5)
    fig.set_dpi(100)
    fig.tight_layout()
    # save_name = "MSER_eq_vol.png"
    # fig.savefig(job.fn(save_name), dpi=300, facecolor="white")
    # plt.close(fig)

def check_equil_converge(eq_data_dict, prod_tol):
    equil_matrix = []
    res_matrix = []
    prod_begin = 0
    prop_names = list(eq_data_dict.keys())
    num_cols = len(prop_names)

    try:
        # Load data for both boxes
        for key in list(eq_data_dict.keys()):
            eq_col = eq_data_dict[key]["data"]
            print("len eq col", len(eq_col))
            prod_tol = len(eq_col)/4
            print("prod tol", prod_tol)
            batch_size = max(1, int(len(eq_col) * 0.0005))

            # Try with ADF test enabled, fallback without it if it fails
            try:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=True,
                    uncertainty="uSD",
                    print_results=False,
                )
                adf_test_failed = results["critical_values"]["1%"] <= results["adf"]
            except:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=False,
                    uncertainty="uSD",
                    print_results=False,
                )
                results["adf"], results["critical_values"], adf_test_failed = (
                    None,
                    None,
                    False,
                )

            equilibrium = len(eq_col) - results["t0"] >= prod_tol
            print("len eq_col", len(eq_col),  "res t0", results["t0"])
            print("eq_col[results[t0]]", eq_col[results["t0"]])
            prod_begin = results["t0"]
            print("Results prod",results["t0"])
            equil_matrix.append(equilibrium and not adf_test_failed)
            res_matrix.append(results)

        for i, is_equilibrated in enumerate(equil_matrix):
            key_name = list(eq_data_dict.keys())[i]
            col_vals = eq_data_dict[key_name]["data"]
            t_vals = eq_data_dict[key_name]["time_data"]
            print(t_vals.min(), t_vals.max())
            # plot all

            # if not all(equil_matrix):
            plot_res_pymser(
                t_vals, col_vals, res_matrix[i], prop_names[i % num_cols]
            )

            # Display outcome
            prod_cycles = len(col_vals) - res_matrix[i]["t0"]
            if is_equilibrated:
                # Plot successful equilibration
                statement = f"       > Success! Found {prod_cycles} production cycles."
            else:
                # Plot failed equilibration
                statement = f"       > Equil Failure! "
                if res_matrix[i]["adf"] is None:
                    # Note: ADF test failed to complete
                    statement += f"ADF test failed to complete! "
                elif res_matrix[i]["adf"] > res_matrix[i]["critical_values"]["1%"]:
                    adf, one_pct = (
                        res_matrix[i]["adf"],
                        res_matrix[i]["critical_values"]["1%"],
                    )
                    statement += f"ADF value: {adf}, 99% confidence value: {one_pct}! "
                if len(col_vals) - res_matrix[i]["t0"] < prod_tol:
                    statement += f"Only {prod_cycles} production cycles found."
            print(statement)
            print("All matrix eq", all(equil_matrix))
            return prod_begin
            # with open("Equil_Output.txt", "a") as f:
            #     print(statement, file=f)

    except Exception as e:
        # This will cause an error in the GEMC operation which lets us know that the job failed
        raise Exception("Error processing job ")

    



In [ ]:
# print(df["Time"].values)
prod_begin = check_equil_converge(eq_data_dict, prod_tol = None)
df = panedr.edr_to_df(file)
volume = df[property].values
vol_prod = volume[prod_begin:]
print(np.mean(vol_prod))

In [ ]:
from thermo import Chemical
from thermo import *
import unyt as u
from utils.molec_class_files import esolvs

# Load class properies for each training molecule
name = "DMF"
mol_names = [name]  # , "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC"]
molec_dict = esolvs.make_dict(mol_names)
molec_data = molec_dict[name]
# from thermo.eos import PengRobinson

# Define the compound (e.g., 'methane', 'ethane', etc.)
smiles_string =  molec_data.smiles_str # Replace with your compound of interest
T = list(molec_data.expt_Pvap.keys())[0]  # Temperature in Kelvin (adjust as needed)
P = list(molec_data.expt_Pvap.values())[0] # Pressure in Pascals (adjust as needed)
P = float((P*u.bar).in_units(u.Pa).value)
# Create a Chemical object for the compound
constants = Chemical(smiles_string)
# print(constants)
# eos_kwargs = {'Pc': constants.Pc, 'Tc': constants.Tc, 'omega': constants.omega}

eos = PR(constants.Tc, constants.Pc, constants.omega, T=T, P=P)

# Use the Peng-Robinson EOS to calculate the vapor density
molar_vol_g = eos.V_g
vapor_density = molec_data.molecular_weight/1000/ molar_vol_g
# Print the vapor density
print(f'PR Vapor Density of {constants.name} at {T} K and {P} Pa: rho_v = {vapor_density} kg/m^3')

unit_T = T * u.K
mw = molec_data.molecular_weight * (u.g / u.mol)
R = 8.314 * (u.Pa * u.m**3) / (u.mol * u.K)
vapor_density = float(
    ((molec_data.expt_Pvap[T] * u.bar * mw) / (R * unit_T)).in_units("kg/m**3").value
)
print(f'IG Vapor Density of {constants.name} at {T} K and {P} Pa: rho_v = {vapor_density} kg/m^3')

In [ ]:
def calc_mass_dens(density):
    #Find the region attributed to liquid density
    mass_dens_z = density[:,1]
    find_liq_slab = find_bulk_liq_index(density)
    range_for_liq_dens = find_liq_slab[0]
    #Calculate the density
    prop_vals = mass_dens_z[range_for_liq_dens] #kg/m^3
    return prop_vals

def find_bulk_liq_index(density):
    #Use np.diff to approximate the 1st derivative
    ES_numdens_z = density[:,1]
    x = density[:,0]
    dy = np.gradient(ES_numdens_z, density[:,0])
    #Use findpeaks to find the peaks and valleys, interpolating to get as close as possible
    fp = findpeaks(lookahead=1, interpolate=10)
    results = fp.fit(dy)["df_interp"]
    all_peaks = results[results['peak'] | results['valley']].index.values
    y_vals = results['y'].iloc[all_peaks]
    peak_index = all_peaks[np.argmax(y_vals)]
    valley_index = all_peaks[np.argmin(y_vals)]
    interfaces = [peak_index, valley_index]

    #Divide the indices by 10 to get the correct index for the density based on interpolation
    interfaces = [int(i/10) for i in interfaces]

    #Get the range of indices for the bulk liquid density regardless of whether interfaces have shifted
    if interfaces[0] < interfaces[1]:
        range_org_liq = list(range(interfaces[0], interfaces[1] + 1))
    else:
        range_org_liq = list(range(interfaces[0], len(results)//10)) + list(range(0, interfaces[1] + 1))
    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[interfaces[0]:interfaces[1]]*max(x)/(len(x)*10), results['y'].iloc[interfaces[0]:interfaces[1]], label='ES', color='red')
    # # plt.scatter(results['x'].iloc[interfaces]*max(x)/(len(x)*10), results['y'].iloc[interfaces], label='ES', color='red')


    # #Divide the indices by 10 to get the correct index for the density based on interpolation
    # interfaces = [int(i/10) for i in interfaces]
    # range_org_liq = np.arange(interfaces[0], interfaces[1], 1)
    #Get the median density from the bulk range
    median_dens_liq = np.median(ES_numdens_z[range_org_liq])

    # Find the first and last index where density >= median
    valid_idx = np.where(ES_numdens_z[range_org_liq] >= median_dens_liq)[0]
    #Cut the range to only include indeces where the density is above the median
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1  # +1 to include the last valid index
        range_for_liq_dens = range_org_liq[start:end]
    else:
        range_for_liq_dens = np.array([], dtype=int)

    return range_for_liq_dens, interfaces, ES_numdens_z

In [ ]:
density = np.loadtxt("density_iters/runs_npzzat_rect/workspace/311f916a543f96c0f1673766035e1783/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
prop_vals = calc_mass_dens(density)

print(np.mean(prop_vals))

In [ ]:
# density = np.loadtxt("density_iters/runs/workspace/da7fe0e92a33f133ad3d366b2972ea9b/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
# block_data = []
# from findpeaks import findpeaks
# for r in [1,2,3]:
#     density = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_{r}.xvg", comments=["#", "@"])

#     prop_vals = calc_mass_dens(density)
#     block_data.append(np.mean(prop_vals))

#     # print(prop_vals)

# var = np.var(block_data, dtype=np.float64)/2
# density2 = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_all.xvg", comments=["#", "@"])

# prop_vals2 = calc_mass_dens(density2)
# avg = np.mean(prop_vals)

# print(avg, var)